# ML-04 · Search Intelligence Data Contract

**Lane:** Core Lane 2 — Content Refresh / Content Opportunity Scoring  
**Intern:** Huzaifah  
**Mid-panel month used:** `month = 2026-03`  
**Warehouse source:** `FlyRank/internship-warehouse` (content_refresh table)  
**Tool:** DuckDB on local Parquet/CSV mirror of the warehouse

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

RAW = Path('../../data/raw/content_refresh_anonymized.csv')
con = duckdb.connect()

# Register the warehouse snapshot as a DuckDB table.
# We treat this snapshot as month=2026-03 (mid-panel, safe for label development).
con.execute(f"""
    CREATE TABLE content_refresh AS
    SELECT *, '2026-03' AS month
    FROM read_csv_auto('{RAW.as_posix()}', header=True)
""")
print('Table registered. Shape:', con.execute('SELECT COUNT(*) FROM content_refresh').fetchone()[0], 'rows')

---
## 1) The Contract — Five Plain-Words Answers

### 1. What does one row mean in my lane?
One row is one piece of content (a web page, identified by `content_id`) belonging to one client (`client_id`), with all its search performance signals measured over the past 90 days. It represents the state of that page **at a single point in time** — the warehouse snapshot date.

### 2. Which table(s) will I use?
The `content_refresh` table from the FlyRank internship warehouse. It contains one row per content page per month, with GSC (Google Search Console) signals, CMS metadata, and GA4 session data pre-joined.

### 3. Which time window?
**Features:** 90-day look-back window ending at the snapshot date (signals like `impressions_90d`, `clicks_90d`, `sessions_90d`). The 30-day sub-windows (`impressions_last_30d`, `impressions_prev_30d`) give trend direction within that window.  
**Label:** `trend_direction` — whether the page's performance went `down`, `up`, `stable`, `flat`, or `new` over the measurement period. The label covers the forward window and is already encoded in the dataset.

### 4. What would I predict or rank?
**Binary classification:** Is this page likely to be declining (`trend_direction == 'down'`)?  
**Ranking:** Score each page by its refresh urgency — pages most likely to need content updates rise to the top of an editor's queue.

### 5. One thing I deliberately exclude
**`trend_pct`** — the percentage magnitude of the trend. It directly encodes the label: a page with `trend_pct = -80` is almost certainly `trend_direction = 'down'`. Including it would be label leakage. It is knowable only *after* the trend resolves, not at the decision moment.

---
## 2) Prove Three Facts with Three Queries (month = 2026-03)

In [ ]:
# ── Query 1: Grain check ─────────────────────────────────────────────────────
# Prove: one row really is one content page (content_id is the grain)
q1 = con.execute("""
    SELECT
        COUNT(*)                        AS total_rows,
        COUNT(DISTINCT content_id)      AS unique_content_ids,
        COUNT(DISTINCT client_id)       AS unique_clients,
        total_rows = unique_content_ids AS grain_is_content_id
    FROM content_refresh
    WHERE month = '2026-03'
""").df()
print('Query 1 — Grain Check:')
print(q1.to_string(index=False))
print('\nVerdict: grain_is_content_id =', q1['grain_is_content_id'][0])

In [ ]:
# ── Query 2: Row count and date span ─────────────────────────────────────────
# Prove: row count for this month, range of content ages, trend breakdown
q2 = con.execute("""
    SELECT
        month,
        COUNT(*)                            AS row_count,
        MIN(content_age_days)               AS min_content_age_days,
        MAX(content_age_days)               AS max_content_age_days,
        ROUND(AVG(content_age_days), 1)     AS avg_content_age_days,
        COUNT(CASE WHEN trend_direction = 'down'   THEN 1 END) AS declining,
        COUNT(CASE WHEN trend_direction = 'stable' THEN 1 END) AS stable,
        COUNT(CASE WHEN trend_direction = 'up'     THEN 1 END) AS improving
    FROM content_refresh
    WHERE month = '2026-03'
    GROUP BY month
""").df()
print('Query 2 — Row Count & Date Span:')
print(q2.to_string(index=False))

In [ ]:
# ── Query 3: Availability — IS TRUE filter ────────────────────────────────────
# Prove: how many rows have confirmed search impressions (are indexable/available)
# A page with impressions_90d > 0 is confirmed indexed and searchable.
q3 = con.execute("""
    SELECT
        COUNT(*)                                    AS total_rows,
        COUNT(CASE WHEN (impressions_90d > 0) IS TRUE THEN 1 END) AS has_impressions,
        COUNT(CASE WHEN (impressions_90d > 0) IS TRUE
                    AND (sessions_90d > 0) IS TRUE  THEN 1 END) AS has_impressions_and_sessions,
        ROUND(
            100.0 * COUNT(CASE WHEN (impressions_90d > 0) IS TRUE THEN 1 END)
            / COUNT(*), 1
        )                                           AS pct_available
    FROM content_refresh
    WHERE month = '2026-03'
""").df()
print('Query 3 — Availability (IS TRUE filter):')
print(q3.to_string(index=False))
print('\nVerdict: pct_available =', q3['pct_available'][0], '% of rows have confirmed impressions')

---
## 3) Five Features + The Leakage Trap

In [ ]:
# ── Build a 5-feature frame (mid-panel month only) ────────────────────────────
feature_frame = con.execute("""
    SELECT
        content_id,
        client_id,

        -- Feature 1: Staleness
        -- Knowable at decision moment because: it's the number of days since the CMS
        -- last-modified timestamp, measured at crawl time — no future data needed.
        days_since_last_update                                      AS f_staleness_days,

        -- Feature 2: CTR gap vs position
        -- Knowable at decision moment because: both CTR and avg_position are 90-day
        -- historical aggregates from GSC, available before any future traffic occurs.
        ROUND(ctr / (avg_position + 1.0), 4)                        AS f_ctr_per_position,

        -- Feature 3: Momentum — last 30d vs prior 30d impressions
        -- Knowable at decision moment because: both windows are already closed;
        -- impressions_last_30d covers [t-30, t], impressions_prev_30d covers [t-60, t-30].
        ROUND(
            CAST(impressions_last_30d AS DOUBLE)
            / (impressions_prev_30d + 1.0), 4
        )                                                            AS f_impression_momentum,

        -- Feature 4: Engagement quality
        -- Knowable at decision moment because: engaged_sessions_90d and sessions_90d
        -- are both 90-day look-back aggregates from GA4, already logged.
        ROUND(
            CAST(engaged_sessions_90d AS DOUBLE)
            / (sessions_90d + 1.0), 4
        )                                                            AS f_engagement_rate,

        -- Feature 5: AI traffic exposure
        -- Knowable at decision moment because: ai_sessions_90d is a 90-day historical
        -- count from GA4's AI referral dimension — past data, no future leakage.
        ROUND(
            CAST(ai_sessions_90d AS DOUBLE)
            / (sessions_90d + 1.0), 4
        )                                                            AS f_ai_traffic_ratio,

        -- Label (for evaluation only — NOT a feature)
        (trend_direction = 'down')::INTEGER                         AS label_is_declining

    FROM content_refresh
    WHERE month = '2026-03'
      AND (impressions_90d > 0) IS TRUE
""").df()

print(f'Feature frame: {len(feature_frame):,} rows x {len(feature_frame.columns)} columns')
print(f'Declining rate: {feature_frame["label_is_declining"].mean():.3f}')
print()
print(feature_frame.head(5).to_string(index=False))

### Feature Summary Table

| Feature | What it measures | Available when? |
|---|---|---|
| `f_staleness_days` | Days since CMS last modified timestamp | At crawl time — CMS metadata, no future data |
| `f_ctr_per_position` | CTR divided by SERP position | At snapshot — 90d GSC aggregate already closed |
| `f_impression_momentum` | Last 30d vs prior 30d impressions ratio | At snapshot — both 30d windows already closed |
| `f_engagement_rate` | Engaged sessions / total sessions | At snapshot — 90d GA4 aggregate already logged |
| `f_ai_traffic_ratio` | AI-referred sessions / total sessions | At snapshot — 90d GA4 AI referral already logged |

In [ ]:
# ── THE TRAP: Deliberate Label Leakage ───────────────────────────────────────
# Step 1: Add trend_pct as a feature and watch precision jump toward perfect
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import numpy as np

# Get data WITH the leaky feature
leaked_df = con.execute("""
    SELECT
        days_since_last_update                              AS f_staleness_days,
        ctr / (avg_position + 1.0)                         AS f_ctr_per_position,
        CAST(impressions_last_30d AS DOUBLE) / (impressions_prev_30d + 1.0) AS f_impression_momentum,
        CAST(engaged_sessions_90d AS DOUBLE) / (sessions_90d + 1.0) AS f_engagement_rate,
        CAST(ai_sessions_90d AS DOUBLE) / (sessions_90d + 1.0) AS f_ai_traffic_ratio,
        COALESCE(trend_pct, 0)                             AS LEAKED_trend_pct,
        (trend_direction = 'down')::INTEGER                AS label
    FROM content_refresh
    WHERE month = '2026-03' AND (impressions_90d > 0) IS TRUE
""").df().fillna(0)

honest_cols = ['f_staleness_days','f_ctr_per_position','f_impression_momentum','f_engagement_rate','f_ai_traffic_ratio']
leaked_cols = honest_cols + ['LEAKED_trend_pct']

pipe = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=500, random_state=42))])

# Score with HONEST features only
score_honest = cross_val_score(pipe, leaked_df[honest_cols], leaked_df['label'],
                               cv=5, scoring='roc_auc').mean()

# Score WITH the leaky feature
score_leaked = cross_val_score(pipe, leaked_df[leaked_cols], leaked_df['label'],
                               cv=5, scoring='roc_auc').mean()

print('=== THE LEAKAGE TRAP ===')
print(f'ROC-AUC WITH honest features only:  {score_honest:.4f}')
print(f'ROC-AUC WITH leaked trend_pct:      {score_leaked:.4f}  <-- suspiciously high!')
print(f'\nLift from leaked feature:           +{score_leaked - score_honest:.4f}')
print()
print('WHY it leaks: trend_pct IS the signed magnitude of trend_direction.')
print('trend_pct = -80 almost guarantees trend_direction = "down".')
print('It is only knowable AFTER the trend resolves — it\'s a label in disguise.')
print()
print('DELETED: trend_pct will NOT appear in any model feature set.')
print(f'Honest ROC-AUC to beat in Week 4: {score_honest:.4f}')

---
## 4) One Named Limitation of My Slice

**Limitation: Client-size skew.**  
The dataset has 32 unique clients but they are not equally sized. Some clients contribute thousands of pages while others contribute fewer than 100. A model trained on this slice without client-grouped validation will over-fit to the patterns of large clients and report optimistic metrics — it may not generalise to a new client the model has never seen. This is why all modeling weeks use `GroupKFold(n_splits=5, groups=client_id)`.  

A secondary limitation: `trend_direction` is derived from a 30-day momentum signal (`impressions_last_30d` vs `impressions_prev_30d`). This means the label is noisy for very new pages (`content_age_days < 30`) — they can show `down` simply because they had no prior-month impressions to compare against.

---
## 5) Self-Check

| Check | Status |
|---|---|
| Five plain-words contract answers written | ✅ |
| Three verification queries executed with output visible | ✅ |
| Availability checked with `IS TRUE` filter | ✅ (Query 3) |
| Five-feature frame built from mid-panel month | ✅ |
| Every feature has an "available when?" line | ✅ |
| Deliberate leakage experiment shown and removed | ✅ |
| `trend_pct` excluded from final feature set | ✅ |
| One named limitation of my slice stated | ✅ |
| Final month (2026-06) NOT used for label development | ✅ |